## Generatore di modello
Da usare su Google Colab per creare il modello. Scegliere se usare GPU o TPU. Il modello e i relativi indici si troveranno nella cartella `model`.

### Per GPU

In [ ]:
!pip install tensorflow

In [ ]:
import numpy as np
import json
import os
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

os.makedirs("model", exist_ok=True)

# =========================================================
# 1. TESTO DI PARTENZA
# =========================================================

with open("divina_commedia.txt", "rt") as f:
    testo = f.read()

# =========================================================
# 2. TOKENIZZAZIONE
# =========================================================

tokenizer = Tokenizer()
tokenizer.fit_on_texts([testo])

sequenza_tokenizzata = tokenizer.texts_to_sequences([testo])[0]
vocabolario = len(tokenizer.word_index) + 1

print("Vocabolario:")
print(tokenizer.word_index)
print()

print("Sequenza tokenizzata:")
print(sequenza_tokenizzata)
print()

# =========================================================
# 3. CREAZIONE DATASET
# =========================================================

lunghezza_sequenza = 5

X = []
y = []

for i in range(len(sequenza_tokenizzata) - lunghezza_sequenza):
    seq_input = sequenza_tokenizzata[i:i + lunghezza_sequenza]
    parola_output = sequenza_tokenizzata[i + lunghezza_sequenza]

    X.append(seq_input)
    y.append(parola_output)

X = np.array(X)
y = np.array(y)

print("Shape X:", X.shape)
print("Shape y:", y.shape)

# one-hot encoding di y
y = to_categorical(y, num_classes=vocabolario)

print("Shape y dopo one-hot:", y.shape)
print()

# =========================================================
# 4. MODELLO
# =========================================================

modello = Sequential([
    Embedding(input_dim=vocabolario, output_dim=32),
    LSTM(64),
    Dense(vocabolario, activation="softmax")
])

modello.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

modello.summary()

# =========================================================
# 5. TRAINING
# =========================================================

# callback che blocca l'effetto del overfitting, quando il loss rimane uguale per 4 epoche
early_stopping = EarlyStopping(monitor="loss",
                               patience=4,
                               restore_best_weights=True)

modello.fit(X, y, epochs=300, verbose=1, callbacks=[early_stopping])
modello.save("model/model.keras")

# =========================================================
# 6. MAPPA INVERSA indice -> parola
# =========================================================

indice_parola = {indice: parola for parola, indice in tokenizer.word_index.items()}

with open("model/indices.json", "wt") as f:
    json.dump(indice_parola, f)

### Per TPU

In [ ]:
!pip install tensorflow==2.18.0 tensorflow-tpu==2.18.0 --find-links=https://storage.googleapis.com/libtpu-tf-releases/index.html
!pip install ml_dtypes==0.5.0 --force

In [ ]:
import numpy as np
import json
import os

from tensorflow.config import experimental_connect_to_cluster
from tensorflow.tpu.experimental import initialize_tpu_system
from tensorflow import distribute
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

os.makedirs("model", exist_ok=True)

# per sfruttare le tpu necessitiamo di specificare di star usando le tpu
resolver = distribute.cluster_resolver.TPUClusterResolver(tpu='local')
experimental_connect_to_cluster(resolver)
initialize_tpu_system(resolver)

strategy = distribute.TPUStrategy(resolver)

with strategy.scope():
    # =========================================================
    # 1. TESTO DI PARTENZA
    # =========================================================

    with open("divina_commedia.txt", "rt") as f:
        testo = f.read()

    # =========================================================
    # 2. TOKENIZZAZIONE
    # =========================================================

    tokenizer = Tokenizer()
    tokenizer.fit_on_texts([testo])

    sequenza_tokenizzata = tokenizer.texts_to_sequences([testo])[0]
    vocabolario = len(tokenizer.word_index) + 1

    print("Vocabolario:")
    print(tokenizer.word_index)
    print()

    print("Sequenza tokenizzata:")
    print(sequenza_tokenizzata)
    print()

    # =========================================================
    # 3. CREAZIONE DATASET
    # =========================================================

    lunghezza_sequenza = 5

    X = []
    y = []

    for i in range(len(sequenza_tokenizzata) - lunghezza_sequenza):
        seq_input = sequenza_tokenizzata[i:i + lunghezza_sequenza]
        parola_output = sequenza_tokenizzata[i + lunghezza_sequenza]

        X.append(seq_input)
        y.append(parola_output)

    X = np.array(X)
    y = np.array(y)

    print("Shape X:", X.shape)
    print("Shape y:", y.shape)

    # one-hot encoding di y
    y = to_categorical(y, num_classes=vocabolario)

    print("Shape y dopo one-hot:", y.shape)
    print()

    # =========================================================
    # 4. MODELLO
    # =========================================================

    modello = Sequential([
        Embedding(input_dim=vocabolario, output_dim=32),
        LSTM(64),
        Dense(vocabolario, activation="softmax")
    ])

    modello.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

    modello.summary()

    # =========================================================
    # 5. TRAINING
    # =========================================================

    # callback che blocca l'effetto del overfitting, quando il loss rimane uguale per 4 epoche
    early_stopping = EarlyStopping(monitor="loss",
                                   patience=4,
                                   restore_best_weights=True)

    modello.fit(X, y, epochs=300, verbose=1, callbacks=[early_stopping])
    modello.save("model/model.keras")

    # =========================================================
    # 6. MAPPA INVERSA indice -> parola
    # =========================================================

    indice_parola = {indice: parola for parola, indice in tokenizer.word_index.items()}

    with open("model/indices.json", "wt") as f:
        json.dump(indice_parola, f)